# Vertical Mode — street elevation and climb candidates

This notebook ranks pedestrian streets by mean altitude across the **Madrid municipality**, then creates a second ranking within a 5 km straight-line radius of Calle de Orense, 70.

- `elev_mean` is the street's average altitude; it does **not** measure climbing.
- `ascent_best_direction_m` estimates uphill gain along a segment in whichever direction climbs more.
- Public stairs and pedestrian paths are listed separately as event candidates.
- The notebook uses every matching `PNOA_MDT05_ETRS89_HU30_*_LID.tif` file in the project folder. The currently available single tile covers only part of Madrid, so results are limited to DEM coverage. Add the other tiles covering Madrid for city-wide rankings and complete 5 km coverage.
- The Madrid boundary is fetched from OpenStreetMap through OSMnx, so the graph cell needs internet access.

In [1]:
from contextlib import ExitStack
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import rasterio
from pyrosm import OSM
from rasterio.io import MemoryFile
from rasterio.merge import merge
from shapely.geometry import MultiLineString, Point, box
from shapely.ops import unary_union

In [2]:
# Load the walking network from the local Geofabrik extract.
PBF_PATH = "madrid-261103.osm.pbf"
osm = OSM(PBF_PATH)
edges = osm.get_network(
    network_type="walking",
    extra_attributes=["incline", "step_count", "ramp", "handrail", "lit", "surface"],
    nodes=False,
)

# The PBF includes more than the Madrid municipality. Fetch its boundary
# and keep only pedestrian edges inside it.
CITY_QUERY = "Madrid, Spain"
city_boundary = ox.geocode_to_gdf(CITY_QUERY).to_crs("EPSG:25830")
if city_boundary.empty:
    raise ValueError(f"Could not find a boundary for {CITY_QUERY!r}")
if edges.crs is None:
    raise ValueError("The walking graph has no CRS; cannot measure distances safely.")

edges = edges.to_crs("EPSG:25830")
edges["edge_id"] = np.arange(len(edges), dtype=np.int64)
madrid_boundary = city_boundary[["geometry"]].dissolve()
edges_madrid = gpd.clip(edges, madrid_boundary).reset_index(drop=True)

print(f"Edges in PBF: {len(edges):,}")
print(f"Edges clipped to Madrid municipality: {len(edges_madrid):,}")

Edges in PBF: 367,245
Edges clipped to Madrid municipality: 145,353


In [4]:
# Read and mosaic all matching 5 m PNOA tiles in the project folder.
RASTER_GLOB = "PNOA_MDT05_ETRS89_HU30_*_LID.tif"
RASTER_PATHS = sorted(Path(".").glob(RASTER_GLOB))
if not RASTER_PATHS:
    raise FileNotFoundError(f"No elevation rasters match {RASTER_GLOB!r}")

with ExitStack() as stack:
    raster_sources = [stack.enter_context(rasterio.open(path)) for path in RASTER_PATHS]
    first_source = raster_sources[0]
    if any(src.crs != first_source.crs for src in raster_sources):
        raise ValueError("All elevation tiles must use the same CRS.")
    if any(src.nodata != first_source.nodata for src in raster_sources):
        raise ValueError("All elevation tiles must use the same nodata value.")
    if any(src.res != first_source.res for src in raster_sources):
        raise ValueError("All elevation tiles must use the same pixel resolution.")

    dem_nodata = first_source.nodata
    dem_crs = first_source.crs
    dem_tiles_footprint = [box(*src.bounds) for src in raster_sources]
    dem_data, dem_transform = merge(raster_sources, nodata=dem_nodata)

dem_footprint = unary_union(dem_tiles_footprint)
dem_memory_file = MemoryFile()
dem = dem_memory_file.open(
    driver="GTiff",
    height=dem_data.shape[1],
    width=dem_data.shape[2],
    count=1,
    dtype=dem_data.dtype,
    crs=dem_crs,
    transform=dem_transform,
    nodata=dem_nodata,
)
dem.write(dem_data)

city_area = madrid_boundary.geometry.area.sum()
city_dem_coverage_pct = (
    dem_footprint.intersection(madrid_boundary.geometry.iloc[0]).area
    / city_area * 100
)
print(f"Elevation tiles loaded: {len(RASTER_PATHS)}")
print(f"Madrid municipality area covered by tile footprints: {city_dem_coverage_pct:.1f}%")
if city_dem_coverage_pct < 99:
    print("WARNING: Madrid-wide rankings only cover streets inside the supplied DEM tiles.")

Elevation tiles loaded: 1
Madrid municipality area covered by tile footprints: 64.3%


In [7]:
def sample_elevation_metrics(lines, raster, spacing_m=10, batch_size=2000):
    """Sample a DEM along lines and compute altitude plus directional ascent."""
    if spacing_m <= 0:
        raise ValueError("spacing_m must be positive")

    def extract_line_parts(geometry):
        if geometry is None or geometry.is_empty:
            return []
        if geometry.geom_type == "LineString":
            return [geometry]
        if geometry.geom_type in {"MultiLineString", "GeometryCollection"}:
            return [
                part
                for child in geometry.geoms
                for part in extract_line_parts(child)
            ]
        return []

    parts_by_edge = [extract_line_parts(geometry) for geometry in lines.geometry]
    keep = [index for index, parts in enumerate(parts_by_edge) if parts]
    result = lines.iloc[keep].copy()
    parts_by_edge = [parts_by_edge[index] for index in keep]
    normalized_geometries = [
        parts[0] if len(parts) == 1 else MultiLineString(parts)
        for parts in parts_by_edge
    ]
    result = result.set_geometry(
        gpd.GeoSeries(normalized_geometries, index=result.index, crs=lines.crs)
    )

    metric_names = [
        "elev_mean",
        "elev_min",
        "elev_max",
        "ascent_forward_m",
        "ascent_reverse_m",
        "ascent_best_direction_m",
    ]
    metrics = np.full((len(result), len(metric_names)), np.nan, dtype=float)

    for batch_start in range(0, len(parts_by_edge), batch_size):
        batch = parts_by_edge[batch_start:batch_start + batch_size]
        coordinates = []
        ranges = []

        for parts in batch:
            part_ranges = []
            for part in parts:
                if part.length <= 0:
                    continue
                sample_count = max(2, int(np.ceil(part.length / spacing_m)) + 1)
                start = len(coordinates)
                for distance in np.linspace(0, part.length, sample_count):
                    point = part.interpolate(float(distance))
                    coordinates.append((point.x, point.y))
                part_ranges.append((start, len(coordinates)))
            ranges.append(part_ranges)

        if not coordinates:
            continue

        sampled = np.ma.asarray(list(raster.sample(coordinates, masked=True)))
        values = sampled.filled(np.nan).astype(float).reshape(-1)

        for local_index, value_range in enumerate(ranges):
            part_profiles = [values[start:end] for start, end in value_range]
            valid_parts = [profile[np.isfinite(profile)] for profile in part_profiles]
            valid_parts = [profile for profile in valid_parts if profile.size]
            if not valid_parts:
                continue

            valid = np.concatenate(valid_parts)
            ascent_forward = 0.0
            ascent_reverse = 0.0
            for profile in part_profiles:
                differences = np.diff(profile)
                differences = differences[np.isfinite(differences)]
                ascent_forward += np.maximum(differences, 0).sum()
                ascent_reverse += np.maximum(-differences, 0).sum()
            metrics[batch_start + local_index] = [
                valid.mean(),
                valid.min(),
                valid.max(),
                ascent_forward,
                ascent_reverse,
                max(ascent_forward, ascent_reverse),
            ]

    for column_index, column_name in enumerate(metric_names):
        result[column_name] = metrics[:, column_index]
    result["length_m"] = result.geometry.length
    return result


def add_midpoint_coordinates(lines):
    """Add longitude and latitude columns for mapping/export."""
    def midpoint(geometry):
        parts = [geometry] if geometry.geom_type == "LineString" else list(geometry.geoms)
        target = sum(part.length for part in parts) / 2
        for part in parts:
            if target <= part.length:
                return part.interpolate(target)
            target -= part.length
        return parts[-1].interpolate(parts[-1].length)

    midpoints = gpd.GeoSeries(
        [midpoint(geometry) for geometry in lines.geometry],
        index=lines.index,
        crs=lines.crs,
    ).to_crs("EPSG:4326")
    result = lines.copy()
    result["longitude"] = midpoints.x.to_numpy()
    result["latitude"] = midpoints.y.to_numpy()
    return result

In [9]:
# Global ranking for the part of Madrid covered by the available DEM tiles.
from shapely.geometry import MultiLineString

dem_area = gpd.GeoDataFrame(geometry=[dem_footprint], crs=dem.crs)
edges_in_dem = gpd.clip(edges_madrid, dem_area)
edges_with_elevation = sample_elevation_metrics(
    edges_in_dem,
    dem,
    spacing_m=10,
)
edges_with_elevation = edges_with_elevation.dropna(subset=["elev_mean"])
edges_with_elevation = add_midpoint_coordinates(edges_with_elevation)

top_altitude_madrid = edges_with_elevation.sort_values(
    "elev_mean",
    ascending=False,
).reset_index(drop=True)

print(f"Streets with valid elevation samples: {len(top_altitude_madrid):,}")

Streets with valid elevation samples: 139,028


In [10]:
# Local analysis: the 5 km radius is straight-line distance from this point.
# The coordinates are an OSM geocoded point for house number 70; adjust if
# you have the exact event-door coordinates.
START_LAT = 40.4620695
START_LON = -3.6937063
RADIUS_M = 5000

start_point = gpd.GeoSeries(
    [Point(START_LON, START_LAT)],
    crs="EPSG:4326",
).to_crs(edges_madrid.crs).iloc[0]
radius_area = start_point.buffer(RADIUS_M)
radius_dem_coverage_pct = (
    radius_area.intersection(dem_footprint).area / radius_area.area * 100
)
local_area = radius_area.intersection(dem_footprint)
print(f"5 km circle covered by the supplied DEM tiles: {radius_dem_coverage_pct:.1f}%")
if radius_dem_coverage_pct < 99:
    print("WARNING: the 5 km ranking is incomplete until the missing DEM tiles are added.")

local_area_gdf = gpd.GeoDataFrame(geometry=[local_area], crs=edges_madrid.crs)
local_edges = gpd.clip(edges_madrid, local_area_gdf)
local_with_elevation = sample_elevation_metrics(
    local_edges,
    dem,
    spacing_m=5,
).dropna(subset=["elev_mean"])
local_with_elevation = add_midpoint_coordinates(local_with_elevation)

top_altitude_5km = local_with_elevation.sort_values(
    "elev_mean",
    ascending=False,
).reset_index(drop=True)

top_climb_candidates_5km = local_with_elevation.sort_values(
    "ascent_best_direction_m",
    ascending=False,
).reset_index(drop=True)

blocked_access = {"no", "private"}
access_ok = ~local_with_elevation.get(
    "access", pd.Series("", index=local_with_elevation.index)
).fillna("").astype(str).str.lower().isin(blocked_access)
foot_ok = ~local_with_elevation.get(
    "foot", pd.Series("", index=local_with_elevation.index)
).fillna("").astype(str).str.lower().isin(blocked_access)
highway = local_with_elevation.get(
    "highway", pd.Series("", index=local_with_elevation.index)
).fillna("").astype(str).str.lower()

public_stairs_5km = local_with_elevation.loc[
    highway.eq("steps") & access_ok & foot_ok
].sort_values("ascent_best_direction_m", ascending=False).reset_index(drop=True)
pedestrian_climb_features_5km = local_with_elevation.loc[
    highway.isin(["steps", "footway", "path", "pedestrian"]) & access_ok & foot_ok
].sort_values("ascent_best_direction_m", ascending=False).reset_index(drop=True)

print(f"Pedestrian edges in the DEM-covered 5 km area: {len(local_with_elevation):,}")
print(f"Mapped public-access stair candidates: {len(public_stairs_5km):,}")

5 km circle covered by the supplied DEM tiles: 97.5%
Pedestrian edges in the DEM-covered 5 km area: 42,370
Mapped public-access stair candidates: 2,199


In [11]:
display_columns = [
    "edge_id", "name", "highway", "elev_mean", "ascent_best_direction_m",
    "longitude", "latitude",
]
display_columns = [c for c in display_columns if c in top_altitude_madrid.columns]
print("Highest streets by mean altitude (within supplied DEM coverage):")
print(top_altitude_madrid[display_columns].head(20).to_string(index=False))

Highest streets by mean altitude (within supplied DEM coverage):
 edge_id                     name highway  elev_mean  ascent_best_direction_m  longitude  latitude
  330478                      NaN footway 741.568970                 0.000000  -3.688435 40.493957
  330479                      NaN footway 741.488495                 0.160950  -3.688444 40.493979
  330480                      NaN footway 741.375519                 0.065002  -3.688455 40.494005
  346475                      NaN footway 740.688391                 1.958008  -3.688797 40.493907
  285410                      NaN footway 739.978786                 4.924988  -3.688425 40.494229
  285441                      NaN footway 739.966003                 0.000000  -3.688303 40.494233
  285435                      NaN footway 739.881500                 0.237000  -3.688282 40.494194
  285436                      NaN footway 739.864502                 0.203003  -3.688293 40.494214
  282912                      NaN footway 73

In [12]:
local_display_columns = [
    "edge_id", "name", "highway", "elev_mean", "ascent_best_direction_m",
    "longitude", "latitude",
]
local_display_columns = [c for c in local_display_columns if c in top_altitude_5km.columns]
stairs_display_columns = local_display_columns + [
    c for c in ["incline", "step_count", "ramp", "handrail", "access", "foot"]
    if c in public_stairs_5km.columns and c not in local_display_columns
]

print("Highest streets within 5 km by mean altitude:")
print(top_altitude_5km[local_display_columns].head(20).to_string(index=False))

print("\nBest pedestrian climb candidates within 5 km:")
print(pedestrian_climb_features_5km[local_display_columns].head(20).to_string(index=False))

print("\nMapped stairways within 5 km:")
print(public_stairs_5km[stairs_display_columns].head(20).to_string(index=False))

Highest streets within 5 km by mean altitude:
 edge_id                     name highway  elev_mean  ascent_best_direction_m  longitude  latitude
  330478                      NaN footway 741.568970                 0.000000  -3.688435 40.493957
  330479                      NaN footway 741.488495                 0.160950  -3.688444 40.493979
  330480                      NaN footway 741.375519                 0.065002  -3.688455 40.494005
  346475                      NaN footway 740.826866                 2.015015  -3.688797 40.493907
  285410                      NaN footway 740.100925                 6.382935  -3.688425 40.494229
  285441                      NaN footway 739.966003                 0.000000  -3.688303 40.494233
  285435                      NaN footway 739.881500                 0.237000  -3.688282 40.494194
  285436                      NaN footway 739.864502                 0.203003  -3.688293 40.494214
  282912                      NaN footway 739.750000           

In [13]:
output_columns = [
    "edge_id", "id", "name", "highway", "access", "foot", "surface",
    "elev_mean", "elev_min", "elev_max", "ascent_forward_m",
    "ascent_reverse_m", "ascent_best_direction_m", "length_m",
    "longitude", "latitude", "geometry",
]
output_columns = [c for c in output_columns if c in edges_with_elevation.columns]
edges_with_elevation[output_columns].to_file(
    "calles_altitud_madrid.gpkg",
    driver="GPKG",
)

top_altitude_madrid[display_columns].to_csv("top_altitude_madrid.csv", index=False)
top_altitude_5km[local_display_columns].to_csv("top_altitude_5km_orense.csv", index=False)
pedestrian_climb_features_5km[local_display_columns].to_csv(
    "climb_candidates_5km_orense.csv", index=False
)
public_stairs_5km[stairs_display_columns].to_csv(
    "stairs_5km_orense.csv", index=False
)
print("Saved the GeoPackage and four CSV rankings.")
dem.close()
dem_memory_file.close()

Saved the GeoPackage and four CSV rankings.
